# 🚀 PrismPrice: End-to-End Multimodal Pricing Pipeline, Optuna HPO & Empirical Report Generator
### Kaggle Notebook Execution Guide
This notebook seamlessly connects with the [A_ML_25 GitHub Repository](https://github.com/arpitkumar2004/A_ML_25.git) to execute:
1. **Automatic Repository Synchronization**: Clones repo and installs all required dependencies including Optuna, LightGBM, XGBoost, CatBoost.
2. **5 Domain Pricing Feature Engineering Sets**: Sub-linear quantity elasticity ($\text{Quantity}^{0.75}$), mass density ratios, unit per-pack ratios, text complexity signals, and missing image indicators.
3. **Optuna Bayesian Hyperparameter Optimization (TPE Sampler)**: Executes Bayesian HPO across LightGBM, XGBoost, CatBoost, ExtraTrees, and Ridge hyperparameters.
4. **5-Fold Cross-Validation Model Suite & Stacking**: Trains 6 base models and fits an out-of-fold `RidgeCV` meta-learner with automated L2 regularization tuning.
5. **15 Publication Figures & 7 JSON/CSV Reports**: Auto-generates all empirical plots (`docs/`) and metrics (`experiments/reports/`).
6. **1-Click Artifact Zipping**: Archives all generated reports, plots, and test predictions into `experiments_reports_and_docs.zip` for instant download back into your local workspace.

In [ ]:
# Step 1: Clone GitHub Repository & Set Up Working Directory
import os
import sys
import subprocess

REPO_URL = "https://github.com/arpitkumar2004/A_ML_25.git"
REPO_DIR = "A_ML_25"

if not os.path.exists(REPO_DIR):
    print(f"📥 Cloning GitHub repository from {REPO_URL}...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir(REPO_DIR)
else:
    print(f"📂 Repository directory '{REPO_DIR}' already exists. Navigating into directory...")
    os.chdir(REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"✅ Current Working Directory: {os.getcwd()}")

In [ ]:
# Step 2: Install Python Dependencies Including Optuna & Boosting Frameworks
!pip install -q scipy scikit-learn lightgbm xgboost catboost optuna matplotlib seaborn joblib pydantic fastapi

In [ ]:
# Step 3: Dataset Auto-Detection in Kaggle Environment
import glob

train_candidates = glob.glob("/kaggle/input/**/train.csv", recursive=True) + glob.glob("data/raw/train.csv") + glob.glob("train.csv")
test_candidates  = glob.glob("/kaggle/input/**/test.csv", recursive=True) + glob.glob("data/raw/test.csv") + glob.glob("test.csv")

train_path = train_candidates[0] if train_candidates else None
test_path  = test_candidates[0] if test_candidates else None

print(f"🔍 Training Dataset Path: {train_path if train_path else 'NOT FOUND (Synthetic Fallback Active)'}")
print(f"🔍 Testing Dataset Path:  {test_path if test_path else 'NOT FOUND'}")

In [ ]:
# Step 4: Execute End-to-End Multimodal Report Generator, Optuna HPO & Benchmark Suite
cmd = [sys.executable, "main.py", "generate-report"]
if train_path:
    cmd.extend(["--data", train_path])

print(f"⚡ Running End-to-End Pipeline: {' '.join(cmd)}")
subprocess.run(cmd, check=True)

In [ ]:
# Step 5: Display Optuna Hyperparameter Optimization Study Results
import json

hpo_file = "experiments/reports/hpo_optuna_results.json"
if os.path.exists(hpo_file):
    with open(hpo_file, "r") as f:
        hpo_data = json.load(f)
    print("🎯 Optuna Bayesian HPO Study Results:")
    print(f"  - Total Trials Executed: {hpo_data.get('total_trials_executed')}")
    print(f"  - Best Trial SMAPE (%):  {hpo_data.get('best_trial_smape'):.4f}%")
    print(f"  - Best Hyperparameters: {json.dumps(hpo_data.get('best_hyperparameters'), indent=4)}")
else:
    print("⚠️ HPO file not found.")

In [ ]:
# Step 6: Fit Final Stacker Ensemble & Save Model Artifacts
if train_path:
    print("🏋️ Fitting final OOF Stacker Meta-Learner on full dataset...")
    cmd_train = [sys.executable, "main.py", "train", "--data", train_path, "--model", "stacker"]
    subprocess.run(cmd_train, check=True)

if test_path and os.path.exists(test_path):
    print("🔮 Generating test predictions submission.csv...")
    cmd_predict = [sys.executable, "main.py", "predict", "--data", test_path, "--output", "submission.csv"]
    subprocess.run(cmd_predict, check=True)
    print("✅ Created submission.csv!")

In [ ]:
# Step 7: Verify All 15 Generated Publication Figures & Reports
print("📊 Generated Metric Reports in experiments/reports/")
reports = glob.glob("experiments/reports/*.json") + glob.glob("experiments/reports/*.csv")
for r in sorted(reports):
    print(f"  - {r}")

print("\n🖼️ Generated Publication Figures in docs/")
figures = glob.glob("docs/*.png")
for f in sorted(figures):
    print(f"  - {f}")

In [ ]:
# Step 8: Zip All Reports & Plots for 1-Click Download
import zipfile

zip_filename = "experiments_reports_and_docs.zip"
print(f"📦 Zipping all generated reports and figures into '{zip_filename}'...")

with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder in ["experiments/reports", "docs"]:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    full_p = os.path.join(root, file)
                    rel_p = os.path.relpath(full_p, os.getcwd())
                    zipf.write(full_p, rel_p)

    if os.path.exists("submission.csv"):
        zipf.write("submission.csv", "submission.csv")

print(f"🎉 DONE! Download '{zip_filename}' from your Kaggle notebook output pane.")